In [1]:
import numpy as np
import pandas as pd

In [61]:
df_balance = pd.read_csv(r"D:\studystudy\Benford\rawdata\3tables\FS_Combas.csv")
df_profit = pd.read_csv(r"D:\studystudy\Benford\rawdata\3tables\FS_Comins.csv")
df_cashflow = pd.read_csv(r"D:\studystudy\Benford\rawdata\3tables\FS_Comscfd.csv")

C:\Users\MLTZ\AppData\Local\Temp\ipykernel_34392\3464925607.py:1: DtypeWarning: Columns (5) have mixed types. Specify dtype option on import or set low_memory=False.
  df_balance = pd.read_csv(r"D:\studystudy\Benford\rawdata\3tables\FS_Combas.csv")


In [62]:
df_balance_desc = pd.read_csv(r"D:\studystudy\Benford\rawdata\3tables\FS_Combas[DES][csv].txt",sep=" - ",header=None)
df_profit_desc = pd.read_csv(r"D:\studystudy\Benford\rawdata\3tables\FS_Comins[DES][csv].txt",sep=" - ",header=None)
df_cashflow_desc = pd.read_csv(r"D:\studystudy\Benford\rawdata\3tables\FS_Comscfd[DES][csv].txt",sep=" - ",header=None)

C:\Users\MLTZ\AppData\Local\Temp\ipykernel_34392\3581138822.py:1: ParserWarning: Falling back to the 'python' engine because the 'c' engine does not support regex separators (separators > 1 char and different from '\s+' are interpreted as regex); you can avoid this warning by specifying engine='python'.
  df_balance_desc = pd.read_csv(r"D:\studystudy\Benford\rawdata\3tables\FS_Combas[DES][csv].txt",sep=" - ",header=None)
C:\Users\MLTZ\AppData\Local\Temp\ipykernel_34392\3581138822.py:2: ParserWarning: Falling back to the 'python' engine because the 'c' engine does not support regex separators (separators > 1 char and different from '\s+' are interpreted as regex); you can avoid this warning by specifying engine='python'.
  df_profit_desc = pd.read_csv(r"D:\studystudy\Benford\rawdata\3tables\FS_Comins[DES][csv].txt",sep=" - ",header=None)
C:\Users\MLTZ\AppData\Local\Temp\ipykernel_34392\3581138822.py:3: ParserWarning: Falling back to the 'python' engine because the 'c' engine does not su

In [63]:
def year_quarter_drop_dup_stk(df_yq,time_col):
    out = df_yq.copy()
    out[time_col] = pd.to_datetime(out[time_col], errors="coerce")
    out['Year'] = out[time_col].dt.year.astype("Int64")
    out["Quarter"] = out[time_col].dt.quarter.astype("Int64")
    out.drop(columns=[time_col], inplace = True)
    out.drop_duplicates(subset=['Stkcd',"Year", "Quarter"], keep="last",inplace = True)
    return out

In [64]:
s0 = df_balance_desc[0].astype(str).str.strip()
tmp = (s0.str.extract(r"^(?P<abbr>\S+)\s*\[(?P<cn>[^\]]+)\]\s*$").dropna())
kw = ["是否", "其中", "注", "备注", "说明", "合计", "总计",'差错更正披露日期','报表类型','证券简称']
pat = "|".join(map(lambda x: x.replace("|", r"\|"), kw))
cols_hit = set(tmp.loc[tmp["cn"].astype(str).str.contains(pat, regex=True), "abbr"])
cols_to_drop = [c for c in df_balance.columns if c in cols_hit]
df_balance.drop(columns=cols_to_drop, inplace=True)
df_balance = year_quarter_drop_dup_stk(df_balance , 'Accper')

In [65]:
s0 = df_profit_desc[0].astype(str).str.strip()
tmp = (s0.str.extract(r"^(?P<abbr>\S+)\s*\[(?P<cn>[^\]]+)\]\s*$").dropna())
kw = ["是否", "其中", "注", "备注", "说明", "合计", "总计",'差错更正披露日期','报表类型','证券简称']
pat = "|".join(map(lambda x: x.replace("|", r"\|"), kw))
cols_hit = set(tmp.loc[tmp["cn"].astype(str).str.contains(pat, regex=True), "abbr"])
cols_to_drop = [c for c in df_profit.columns if c in cols_hit]
df_profit.drop(columns=cols_to_drop, inplace=True)
df_profit = year_quarter_drop_dup_stk(df_profit , 'Accper')

In [66]:
s0 = df_cashflow_desc[0].astype(str).str.strip()
tmp = (s0.str.extract(r"^(?P<abbr>\S+)\s*\[(?P<cn>[^\]]+)\]\s*$").dropna())
kw = ["是否", "其中", "注", "备注", "说明", "合计", "总计",'报表类型',
      '差错更正披露日期', "小计", "净额",'余额', "净增加额", "净减少额",'证券简称']
pat = "|".join(map(lambda x: x.replace("|", r"\|"), kw))
cols_hit = set(tmp.loc[tmp["cn"].astype(str).str.contains(pat, regex=True), "abbr"])
cols_to_drop = [c for c in df_cashflow.columns if c in cols_hit]
df_cashflow.drop(columns=cols_to_drop, inplace=True)
df_cashflow = year_quarter_drop_dup_stk(df_cashflow , 'Accper')

In [67]:
#mask_hit = tmp["cn"].astype(str).str.contains(pat, regex=True)
#cn2abbr_hit = dict(zip(tmp.loc[mask_hit, "cn"], tmp.loc[mask_hit, "abbr"]))
#cn2abbr_miss = dict(zip(tmp.loc[~mask_hit, "cn"], tmp.loc[~mask_hit, "abbr"]))

In [73]:
key = ['Stkcd','Year','Quarter']
df_all =  df_balance.merge(df_profit , on = key , how = 'outer')
df_all = df_all.merge(df_cashflow , on = key , how = 'outer')
df_all

,Stkcd,A001101000,A0d1102000,A0b1103000,A0b1104000,A0b1105000,A0f1106000,A001107000,A0f1108000,A001109000,...,C003008000,C003001000,C003003000,C003002000,C003004000,C003005000,C003006000,C003007000,C004000000,C007000000
0,1,8.042000e+08,0.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,1,3.310000e+07,0.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,1,6.320410e+07,0.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,1,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,1,0.000000e+00,0.0,NaN,90814922.4,NaN,2.377393e+09,1.948935e+08,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
304576,920992,1.554007e+08,NaN,NaN,NaN,NaN,NaN,3.504783e+08,NaN,NaN,...,NaN,NaN,NaN,NaN,4117618.6,NaN,9673093.40,288000.0,287865.39,NaN
304577,920992,5.292141e+08,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,4117618.6,NaN,9728612.39,363000.0,509750.08,NaN
304578,920992,9.567810e+07,NaN,NaN,NaN,NaN,NaN,4.287613e+08,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,102600.0,129836.99,NaN
304579,920992,1.385980e+08,NaN,NaN,NaN,NaN,NaN,3.783068e+08,11645.95,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,7738474.72,205200.0,128730.52,NaN


In [78]:
exclude_cols = ["Stkcd", "Year", "Quarter"]

cols = [c for c in df_all.columns if c not in exclude_cols]

x_df = df_all[cols]

valid = x_df.notna() & (x_df != 0)

row_n_valid = valid.sum(axis=1)
keep = row_n_valid >= 50

x_df = x_df.loc[keep]
valid = valid.loc[keep]

np = __import__("numpy")

x = x_df.to_numpy(dtype="float64", copy=False)
x = np.abs(x)

m = valid.to_numpy(copy=False)

x = np.where(m, x, np.nan)

log10x = np.log10(x)
frac = log10x - np.floor(log10x)

digits = np.floor(10 ** frac).astype("int16")  # 1-9 for valid cells, undefined for NaN

# 把无效位置设为 -1，后面计数时不会命中 0-9
digits = np.where(np.isfinite(log10x), digits, -1)

counts = {}
for d in range(10):
    counts[d] = (digits == d).sum(axis=1)

digit_counts = df_all.loc[keep, exclude_cols].copy()
for d in range(10):
    digit_counts[f"lead_{d}"] = counts[d]

digit_counts["n_valid"] = row_n_valid.loc[keep].to_numpy()

C:\Users\MLTZ\AppData\Local\Temp\ipykernel_34392\391986113.py:27: RuntimeWarning: invalid value encountered in cast
  digits = np.floor(10 ** frac).astype("int16")  # 1-9 for valid cells, undefined for NaN


In [79]:
digit_counts

,Stkcd,Year,Quarter,lead_0,lead_1,lead_2,lead_3,lead_4,lead_5,lead_6,lead_7,lead_8,lead_9,n_valid
7,1,1995,2,0,9,13,6,8,5,3,1,1,4,50
13,1,1998,2,0,18,7,6,9,6,1,6,2,1,56
14,1,1998,4,0,19,13,6,4,5,2,5,4,3,61
15,1,1999,1,0,24,13,10,5,3,3,4,3,1,66
16,1,1999,2,0,12,13,7,7,5,2,3,2,1,52
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
304576,920992,2024,3,0,15,15,10,5,3,5,2,7,8,70
304577,920992,2024,4,0,16,16,11,4,7,2,1,5,7,69
304578,920992,2025,1,0,14,9,8,12,4,2,8,2,4,63
304579,920992,2025,2,0,31,6,10,4,3,2,2,2,8,68


In [82]:
digit_counts

,Stkcd,Year,Quarter,lead_0,lead_1,lead_2,lead_3,lead_4,lead_5,lead_6,lead_7,lead_8,lead_9,n_valid,Benford_MAD
7,1,1995,2,0,9,13,6,8,5,3,1,1,4,50,0.044902
13,1,1998,2,0,18,7,6,9,6,1,6,2,1,56,0.035848
14,1,1998,4,0,19,13,6,4,5,2,5,4,3,61,0.020461
15,1,1999,1,0,24,13,10,5,3,3,4,3,1,66,0.025039
16,1,1999,2,0,12,13,7,7,5,2,3,2,1,52,0.030725
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
304576,920992,2024,3,0,15,15,10,5,3,5,2,7,8,70,0.039549
304577,920992,2024,4,0,16,16,11,4,7,2,1,5,7,69,0.042121
304578,920992,2025,1,0,14,9,8,12,4,2,8,2,4,63,0.040520
304579,920992,2025,2,0,31,6,10,4,3,2,2,2,8,68,0.055303


In [80]:
# Benford 理论分布（1–9）
benford_p = np.log10(1 + 1 / np.arange(1, 10))
# 提取 lead_1 ... lead_9
lead_cols = [f"lead_{d}" for d in range(1, 10)]
lead_mat = digit_counts[lead_cols].to_numpy(dtype="float64")
# n_valid
n = digit_counts["n_valid"].to_numpy(dtype="float64").reshape(-1, 1)
# 实际比例
p_hat = lead_mat / n
# MAD
mad = np.mean(np.abs(p_hat - benford_p), axis=1)
digit_counts["Benford_MAD"] = mad

In [81]:
digit_counts["Benford_MAD"].describe()

count    265272.000000
mean          0.033924
std           0.009744
min           0.006063
25%           0.026954
50%           0.033143
75%           0.040063
max           0.094540
Name: Benford_MAD, dtype: float64

In [ ]:
import pandas as pd
import os
import glob
from pathlib import Path
import numpy as np
folder = Path(r"D:\studystudy\Window Dressing\raw data\stk price")
files = sorted(folder.glob("TRD_BwardQuotationMonth*.csv"))
df_all = pd.concat((pd.read_csv(f) for f in files), ignore_index=True)
df_all.drop(columns = ['Filling','CirculatedMarketValue'],inplace = True)
df2 = df_all.copy()
# 1) dtype 一次性处理（尽量用矢量化、避免反复 astype/assign）
df2["CloseDate"] = pd.to_datetime(df2["CloseDate"], errors="coerce")

# TradingMonth：建议统一成 YYYYMM 的整数，比较快也省内存
# 如果原来是 "2024-01" / "202401" / 202401 混杂，这段会尽量规整
tm = df2["TradingMonth"].astype(str).str.replace("-", "", regex=False).str.slice(0, 6)
df2["TradingMonth_int"] = pd.to_numeric(tm, errors="coerce").astype("Int64")

df2["Year"] = df2["CloseDate"].dt.year
df2["Half"] = (df2["CloseDate"].dt.month > 6).astype("int8") + 1  # 1/2

# 2) 一次排序，后面 idx/first/last 都能用
df2 = df2.sort_values(["Symbol", "Year", "Half", "CloseDate"], kind="mergesort")

gkey = ["Symbol", "Year", "Half"]

# 3) 期末：每半年最后交易日
end_idx = df2.groupby(gkey, sort=False)["CloseDate"].idxmax()
end_px = (
    df2.loc[end_idx, ["Symbol", "Year", "Half", "CloseDate", "ClosePrice"]]
       .rename(columns={"CloseDate": "EndDate", "ClosePrice": "EndClose"})
)

# 4) 期初：每半年内最早 TradingMonth 的“该月第一条记录”的 OpenPrice
# 先找每个半年组最小 TradingMonth
min_tm = df2.groupby(gkey, sort=False)["TradingMonth_int"].min()

# 给每行标记：是否属于该组的最小 TradingMonth
df2 = df2.join(min_tm.rename("min_tm"), on=gkey)
mask = df2["TradingMonth_int"].eq(df2["min_tm"])

# 在最小 TradingMonth 子集里，取每组第一条（因为已按 CloseDate 排序）
start_px = (
    df2[mask]
      .groupby(gkey, sort=False)
      .first()[["TradingMonth_int", "OpenPrice"]]
      .reset_index()
      .rename(columns={"TradingMonth_int": "StartMonth", "OpenPrice": "StartOpen"})
)

# 5) 合并 + 计算收益率/得分
half_ret = (
    start_px.merge(end_px, on=gkey, how="inner")
            .assign(
                EndMonth=lambda x: x["EndDate"].dt.to_period("M"),
                HalfReturn=lambda x: x["EndClose"] / x["StartOpen"]
            )
            .sort_values(["Symbol", "Year", "Half"])
)

half_ret["WinnerScore"] = (
    half_ret.groupby(["Year", "Half"], sort=False)["HalfReturn"]
            .rank(pct=True, method="average")
            .mul(5).pipe(np.ceil)
            .clip(1, 5)
            .astype("int8")
)
half_ret['HalfReturn'] = half_ret['HalfReturn'] - 1